# WSJ 2027 - Gruppindelning Direktresa

Assign confirmed direktresa deltagare into groups of exactly 36.

Direktresa participants travel independently to/from WSJ in Poland.
Same grouping algorithm as rundresa but applied to the direktresa subset.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — Hilbert curve sort
3. **Friend wish** — at least ONE friend in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex balance

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

# Fetch and parse all participants
raw_data = u.fetch_participants()
df, skipped = u.build_participant_dataframe(raw_data)

Fetched 1233 participants
Confirmed: 1173, Unconfirmed: 27, Cancelled: 33
Total confirmed participants: 1173
Skipped: 27 unconfirmed, 0 wrong age/no DOB

By category:
category
deltagare    983
ist          168
cmt           22

By travel type:
travel
rundresa      833
direktresa    199
egen_resa     119
other          22

Friend wishes:
  With friend 1 member no: 642
  With friend 2 member no: 363
  With friend 1 name (text): 57
  With friend 2 name (text): 38


In [2]:
GROUP_SIZE = 36

# Filter to direktresa deltagare only
df_direkt = df[df['travel'] == 'direktresa'].copy().reset_index(drop=True)
print(f"=== Direktresa deltagare ===")
print(f"Participants: {len(df_direkt)}")

# Assign coordinates and Hilbert sort
u.assign_coordinates(df_direkt)
df_sorted = u.add_hilbert_index(df_direkt)

n_full = len(df_sorted) // GROUP_SIZE
remainder = len(df_sorted) % GROUP_SIZE
total_groups = n_full + (1 if remainder > 0 else 0)
print(f"\nGroups: {n_full} x {GROUP_SIZE} + 1 x {remainder} = {total_groups} total")
print(f"\nBy region:")
print(df_sorted['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_sorted['age'].value_counts().sort_index().to_string())
print(f"\nBy sex:")
print(df_sorted['sex'].map(u.SEX_MAP).value_counts().to_string())

=== Direktresa deltagare ===
Participants: 199
With coordinates: 196
Without coordinates (Sweden centroid): 3

Groups: 5 x 36 + 1 x 19 = 6 total

By region:
region
Region Stockholm    80
Region Södra        35
Region Norr/Mitt    28
Region Västra       27
Region Östra        19
                     1

By age:
age
14    51
15    58
16    54
17    36

By sex:
sex
Man       104
Kvinna     93
Annat       2


In [3]:
# Resolve text-only friend wishes via fuzzy name matching
u.resolve_friend_wishes(df_sorted, df)
friend_wishes = u.build_friend_graph(df_sorted)

=== Text Friend Name Matching ===
Text-only wishes: 11
Matched & applied: 8
Generic wishes (not a person): 0
Unresolved (friend not in project): 3

Matched:
  Alma Oliver Elgh -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via exact
  Alma Oliver Elgh -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via fuzzy(0.92)+kar
  Armilde Westerblom -> Loke Jageteg (Falkenbergs Scoutkår) [rundresa] via exact
  Armilde Westerblom -> Vera Hedgren (Bohus Scoutkår) [rundresa] via exact
  Astrid Dodd -> Gustav Glimtoft (Älvsjö Scoutkår) [direktresa] via exact
  Elsa Mattsson -> Julia Gunnarsson (Vendelsö Scoutkår) [direktresa] via exact
  Freja Caro -> Karin Hugner (S:t Botvids Scoutkår) [direktresa] via exact
  Rasmus Noring -> Victor Thorburn (Hamre Scoutkår) [rundresa] via exact

Unresolved:
  Elsa Mattsson -> "Dejan Greve Vendelsö scoutkår" (no match found)
  Hilma Wixner -> "Elin Hjertstedt" (no match found)
  Klara Ivarsson -> "Klara Hellberg, Mälarscouterna" (no match found)
=== Final Frie

In [4]:
# Run the full group assignment algorithm
df_sorted = u.assign_groups(df_sorted, GROUP_SIZE, friend_wishes)
total_groups = df_sorted['group'].nunique()

Participants: 199
Groups: 5 x 36 + 1 x 19 = 6 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 88/92
  Kar violations: 2
  Avg geo spread: 2.3607

=== Phase 2: Fix friend wishes ===
  Swaps: 4
  Friend satisfaction: 91/92
  Kar violations: 2
  Avg geo spread: 2.5613

=== Phase 3: Fix kar violations (geo-aware) ===
  Swaps: 2
  Kar violations: 0
  Friend satisfaction: 90/92
  Avg geo spread: 2.5620

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 2
  Friend satisfaction: 92/92
  Kar violations: 0
  Avg geo spread: 2.5627

=== Phase 4: Diversity optimization (geo-penalized) ===
  Swaps: 485
  Diversity score: 17.59 -> 17.86
  Avg geo spread: 2.5627 -> 2.1392

=== FINAL RESULTS ===
Groups: 5 x 36 + 1 x 19
Friend satisfaction: 92/92 (100%)
Kar violations: 0
Total swaps: 493
Diversity: 17.86
Avg geo spread: 2.1392


In [5]:
# Per-group quality metrics
group_of = df_sorted['group'].values
u.print_group_metrics(df_sorted, group_of, total_groups)

Grupp Storlek  Van% MaxKar     M/K/A  14  15  16  17  AvstandKm Karer
--------------------------------------------------------------------------------
    1      36 16/16      4   17/18/1  10  11  10   5        58    21
    2      36 20/20      8   19/16/1  13  12   9   2       132    17
    3      36 17/17      4   17/19/0   8  13   8   7        98    21
    4      36 16/16      4   21/15/0  10   7   9  10       126    22
    5      36 19/19      8   21/15/0   6  11  12   7       103    21
    6      19   4/4      3    9/10/0   4   4   6   5        20    16
--------------------------------------------------------------------------------
Avg geographic spread: 89 km
Min/Max spread: 20 / 132 km


In [6]:
# Export CSV + JSON
OUTPUT_DIR = '/config/notebooks/wsj27'
csv_path, json_path = u.export_results(df_sorted, group_of, total_groups, OUTPUT_DIR, prefix='wsj27_direktresa')

Saved 199 participants to /config/notebooks/wsj27/wsj27_direktresa_grupper.csv
Saved group summary to /config/notebooks/wsj27/wsj27_direktresa_grupper.json

CSV preview (first 10 rows):
 group            name member_no  age    sex                   kar                   district       region friend_1 friend_2       lat       lng
     1   Nora Hultberg   3451529   16 Kvinna        Dalby Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.664664 13.346159
     1   Amos Svensson   3378005   15    Man    Djupadals Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.575069 12.957032
     1     Jamie Arndt   3437949   16    Man    Djupadals Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.575069 12.957032
     1    Miki Lessing   3371249   14  Annat    Djupadals Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.575069 12.957032
     1     Noa Sjögren   3489898   14    Man        Equmenia Scout                       

In [7]:
# Generate interactive map
map_path = f'{OUTPUT_DIR}/wsj_direktresa_karta.html'
u.generate_group_map_html(df_sorted, total_groups, map_path, title='WSJ 2027 Direktresa',
                          friend_wishes=friend_wishes, show_group_arcs=True)
print(f"\nAll outputs:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Map:  {map_path}")

Saved group map: /config/notebooks/wsj27/wsj_direktresa_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/wsj27_direktresa_grupper.csv
  JSON: /config/notebooks/wsj27/wsj27_direktresa_grupper.json
  Map:  /config/notebooks/wsj27/wsj_direktresa_karta.html
